# V51 - No-chat specialist and template-residual challenger

Goal: attack the two slices where the v50 chat-only candidate is weakest. One, users with no chat get near-intercept predictions from an empty TF-IDF row, so an assessment-only specialist overrides them. Two, templated chat carries mostly group signal, so a residual view (user text minus its template centroid) is tested as a separate challenger. Each challenger must clear a paired bootstrap gate against the chat-only baseline or the run fails close.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 20260905
N_SPLITS = 5
GATE_FLOOR = 0.00045
MIN_GROUP = 3
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found in any candidate path"
CHAMPION = next((p for p in [Path("/kaggle/input/intelligo-champion/submission_v42_primary.csv"),
                            Path("results/v42/submission_v42_primary.csv")] if p.exists()), None)
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v51")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve(), "| CHAMPION:", CHAMPION, "| OUT:", OUT.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data | CHAMPION: results/v42/submission_v42_primary.csv | OUT: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/results/v51


## Load data and feature views

Same three views as v50, plus the template-residual view defined in the next cell. Entity disjointness is asserted, not assumed.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
train_ids = set(train["user_id"])
assert not (train_ids & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)

by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
all_users = list(train["user_id"]) + test_ids
is_nochat = np.array([u not in by_user for u in all_users])
print(f"no-chat users: train={is_nochat[:n_train].sum()} test={is_nochat[n_train:].sum()}")

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan

A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
A_tr, A_te = A[:n_train], A[n_train:]

docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3)
T = tfidf.fit_transform(docs).tocsr()
T_tr, T_te = T[:n_train], T[n_train:]
print("assessment:", A.shape, "tfidf:", T.shape)

no-chat users: train=117 test=33


assessment: (5000, 10) tfidf: (5000, 2272)


## Template groups and residual view

Users sharing an exact chat text form a template group. The residual view subtracts the group centroid, keeping only how a user deviates from their template. Small groups keep the raw vector.

In [3]:
def template_group(texts):
    if not texts:
        return "NO_CHAT"
    head = " ".join(str(texts[0]).lower().split()[:6])
    return re.sub(r"\d+", "#", head)

tgroup = np.array([template_group(by_user.get(u, [])) for u in all_users])
uniq, counts = np.unique(tgroup, return_counts=True)
big = set(uniq[(counts >= MIN_GROUP) & (uniq != "NO_CHAT")])
print(f"template groups={len(uniq)} big(>={MIN_GROUP})={len(big)}")

R = T.copy().tolil()
for g in big:
    idx = np.where(tgroup == g)[0]
    centroid = np.asarray(T[idx].mean(0)).ravel()
    for i in idx:
        R[i] = T[i].toarray().ravel() - centroid
R = R.tocsr()
R_tr, R_te = R[:n_train], R[n_train:]
print(f"residual nnz fraction vs raw: {R.nnz / T.nnz:.3f}")

strata = Y.argmax(1)
fold_groups = tgroup[:n_train]

template groups=304 big(>=3)=216


residual nnz fraction vs raw: 5.952


## OOF evaluation of baseline and challengers

Baseline is the v50 chat-only Ridge. Challenger A overrides no-chat users with the assessment view. Challenger B is the residual view. Challenger C applies both. Same folds as v50, so numbers are comparable.

In [4]:
def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(Xtr, Xte, alpha=3.0):
    pred = np.zeros((Xte.shape[0], len(MODS)))
    oof = np.zeros_like(Y)
    fold_score, fold_id = [], np.zeros(len(Xtr), int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        p_va = np.zeros((len(va), len(MODS)))
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha, random_state=SEED)
            m.fit(Xtr[tr], Y[tr, j])
            p_va[:, j] = m.predict(Xtr[va])
            pred[:, j] += m.predict(Xte) / N_SPLITS
        oof[va] = p_va
        fold_score.append(ndcg5(Y[va], p_va).mean())
        fold_id[va] = fi + 1
    return oof, pred, np.array(fold_score), fold_id

Xtr_dense = lambda M: M.toarray() if hasattr(M, "toarray") else M
oof_c, pred_c, f_c, fid = ridge_oof(Xtr_dense(T_tr), Xtr_dense(T_te))
oof_a, pred_a, _, _ = ridge_oof(A_tr, A_te)
oof_r, pred_r, f_r, _ = ridge_oof(Xtr_dense(R_tr), Xtr_dense(R_te))
print(f"baseline chat OOF: {ndcg5(Y, oof_c).mean():.5f} folds={np.round(f_c, 5)}")
print(f"assessment OOF: {ndcg5(Y, oof_a).mean():.5f}")
print(f"residual OOF: {ndcg5(Y, oof_r).mean():.5f} folds={np.round(f_r, 5)}")

nochat_tr = is_nochat[:n_train]
print(f"baseline on train no-chat slice ({nochat_tr.sum()} users): {ndcg5(Y[nochat_tr], oof_c[nochat_tr]).mean():.5f}")
print(f"assessment on train no-chat slice: {ndcg5(Y[nochat_tr], oof_a[nochat_tr]).mean():.5f}")

baseline chat OOF: 0.61629 folds=[0.61169 0.59616 0.65598 0.58344 0.62836]
assessment OOF: 0.41405
residual OOF: 0.53748 folds=[0.53486 0.55051 0.54065 0.53371 0.53344]
baseline on train no-chat slice (117 users): 0.44821
assessment on train no-chat slice: 0.49800


## Gate each challenger and fail close

A challenger is adopted only if its paired bootstrap CI over the baseline clears the floor and its worst validation fold is non-negative. Otherwise the baseline stands.

In [5]:
base = ndcg5(Y, oof_c)
cands = {}
oA = oof_c.copy()
oA[nochat_tr] = oof_a[nochat_tr]
pA = pred_c.copy()
pA[is_nochat[n_train:]] = pred_a[is_nochat[n_train:]]
cands["A no-chat override"] = (oA, pA)
cands["B residual"] = (oof_r, pred_r)
oC = oof_r.copy()
oC[nochat_tr] = oof_a[nochat_tr]
pC = pred_r.copy()
pC[is_nochat[n_train:]] = pred_a[is_nochat[n_train:]]
cands["C residual + override"] = (oC, pC)

winner, winner_pred, winner_name = base, None, "baseline chat (fail-close)"
for name, (oo, pr) in cands.items():
    d = ndcg5(Y, oo) - base
    lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
    wf = min(ndcg5(Y[fid == k], oo[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
    print(f"{name}: delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
    if lo > GATE_FLOOR and wf >= 0 and d.mean() > (winner - base).mean():
        winner, winner_pred, winner_name = ndcg5(Y, oo), pr, name
print("ADOPT:", winner_name, f"OOF={winner.mean():.5f}")

A no-chat override: delta=+0.00146 CI95=[+0.00042,+0.00249] worst-fold=+0.00000
B residual: delta=-0.07881 CI95=[-0.08394,-0.07305] worst-fold=-0.11533
C residual + override: delta=-0.07624 CI95=[-0.08211,-0.07096] worst-fold=-0.11533
ADOPT: baseline chat (fail-close) OOF=0.61629


## Champion disagreement and submission

Same contract as v50: report disagreement when the champion file is present, always write a valid submission.

In [6]:
final_pred = pred_c if winner_pred is None else winner_pred
if CHAMPION is not None:
    ch = pd.read_csv(CHAMPION)
    assert ch["user_id"].tolist() == test_ids
    C = ch[MODS].to_numpy(float)
    t1 = (C.argmax(1) == final_pred.argmax(1)).mean()
    t5 = np.mean([len(set(a) & set(b)) / 5 for a, b in zip(np.argsort(-C, 1)[:, :5], np.argsort(-final_pred, 1)[:, :5])])
    print(f"candidate vs champion: top1 agree={t1:.3f} top5 overlap={t5:.3f}")
else:
    print("champion file absent; skipping disagreement report")
sub = pd.DataFrame(final_pred, columns=MODS)
sub.insert(0, "user_id", test_ids)
sub.to_csv(OUT / "submission_v51_primary.csv", index=False)
print("wrote", OUT / "submission_v51_primary.csv", sub.shape)

candidate vs champion: top1 agree=0.749 top5 overlap=0.810
wrote results/v51/submission_v51_primary.csv (1000, 18)
